# Week 5 · Graph RAG with Neo4j — Game of Thrones (Text-to-Cypher)

**Self-contained notebook** — no external `utils.py` required.

**What you'll see:**
1. A Neo4j **Aura** graph of GOT characters, houses, kills, and marriages — loaded in batches
2. An LLM translating natural language → **Cypher** (the graph query language)
3. A **read-only guardrail** so the LLM can never write to the database *(callback: Week 2 guardrails)*
4. A **self-correcting retry loop** — when generated Cypher fails, the error is fed back to the LLM
5. Hybrid answers combining **graph facts + text context** *(callback: Week 4 vector RAG)*

**Bridge from Week 4:** vector RAG retrieves *similar text*. But "who killed the person married to X?" isn't a similarity question — it's a **relationship traversal**. That's what graphs are for.

> Think of it in database terms: vector RAG is a fuzzy `WHERE description LIKE ...`;
> graph RAG is a chain of `JOIN`s — except the joins (relationships) are first-class citizens.

In [5]:
!pip install neo4j openai --quiet
print("✅  Dependencies installed")

✅  Dependencies installed



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\vivik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import os, json, re
from typing import List, Dict, Any, Optional

# ── Teaching helpers (self-contained, no shared utils) ─────────────
def banner(text: str, char: str = "═", width: int = 70):
    print(f"\n{char * width}\n  {text}\n{char * width}\n")

def section(text: str):
    print(f"\n─── {text} ───\n")

def observe(text: str):
    print(f"💡 OBSERVE: {text}\n")

def discuss(text: str):
    print(f"🗣️  DISCUSS: {text}\n")

print("✅  Helpers ready")

✅  Helpers ready


In [7]:
# ── LLM Client: OpenAI primary, Anthropic fallback, mock last resort ──
class LLMClient:
    """
    Minimal chat client. Tries OpenAI first, then Anthropic, then mock.
    The client object is created ONCE (not per call) — cheaper and faster.
    """
    def __init__(self, openai_model: str = "gpt-4o-mini",
                 anthropic_model: str = "claude-haiku-4-5"):
        self.provider = None
        self.model = None
        self._client = None

        if os.getenv("OPENAI_API_KEY"):
            import openai
            self._client = openai.OpenAI()          # reads key from env
            self.provider, self.model = "openai", openai_model
        elif os.getenv("ANTHROPIC_API_KEY"):
            import anthropic
            self._client = anthropic.Anthropic()    # reads key from env
            self.provider, self.model = "anthropic", anthropic_model
        else:
            self.provider = "mock"
            print("⚠️  No OPENAI_API_KEY / ANTHROPIC_API_KEY found — mock mode.")

    def chat(self, user: str, system: str = "", temperature: float = 0.0) -> str:
        try:
            if self.provider == "openai":
                messages = ([{"role": "system", "content": system}] if system else [])
                messages.append({"role": "user", "content": user})
                resp = self._client.chat.completions.create(
                    model=self.model, messages=messages, temperature=temperature)
                return resp.choices[0].message.content
            if self.provider == "anthropic":
                resp = self._client.messages.create(
                    model=self.model, max_tokens=1024,
                    system=system or "You are a helpful assistant.",
                    temperature=temperature,
                    messages=[{"role": "user", "content": user}])
                return resp.content[0].text
        except Exception as e:
            print(f"❌ LLM error ({self.provider}): {e}")
        return self._mock(user)

    @staticmethod
    def _mock(user: str) -> str:
        if "cypher" in user.lower():
            return ("MATCH (v:Character {name: 'Eddard Stark'})<-[:KILLED]-(k)\n"
                    "RETURN k.name AS killer")
        return "Mock answer (set OPENAI_API_KEY or ANTHROPIC_API_KEY for real responses)."

client = LLMClient()
print(f"✅  LLM ready → provider: {client.provider}, model: {client.model}")

✅  LLM ready → provider: openai, model: gpt-4o-mini


In [10]:
# ── Neo4j Aura Connection ──────────────────────────────────────────
# Reads the exact variable names from an Aura-downloaded .env:
#   NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, NEO4J_DATABASE
# (also accepts NEO4J_USER as a fallback for older setups)
from neo4j import GraphDatabase

NEO4J_URI      = os.getenv("NEO4J_URI", "neo4j+s://27b7456d.databases.neo4j.io")
NEO4J_USER     = os.getenv("NEO4J_USERNAME") or os.getenv("NEO4J_USER") or "neo4j"
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

NEO4J_AVAILABLE = False
driver = None

try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    NEO4J_AVAILABLE = True
    print(f"✅  Connected to Neo4j → {NEO4J_URI} (db: {NEO4J_DATABASE})")
except Exception as e:
    print(f"⚠️  Neo4j not available: {e}")
    print("    Running in MOCK MODE.")
    if "unauthorized" in str(e).lower() or "authentication" in str(e).lower():
        print("    💡 Hint: on Aura the username is almost always 'neo4j',")
        print("       NOT the instance ID. Check NEO4J_USERNAME in your .env.")

def run_cypher(query: str, params: Optional[Dict] = None,
               readonly: bool = True) -> List[Dict]:
    """Execute Cypher and return a list of dicts.
    readonly=True routes through execute_read — the DB itself rejects writes."""
    if not NEO4J_AVAILABLE:
        return []
    def work(tx):
        return [dict(r) for r in tx.run(query, params or {})]
    with driver.session(database=NEO4J_DATABASE) as session:
        return session.execute_read(work) if readonly else session.execute_write(work)

def clear_database():
    """Delete all nodes and relationships (use with caution)."""
    if NEO4J_AVAILABLE:
        run_cypher("MATCH (n) DETACH DELETE n", readonly=False)
        print("🧹 Database cleared.")
    else:
        print("[MOCK] Database would be cleared.")

Unable to retrieve routing information


⚠️  Neo4j not available: Unable to retrieve routing information
    Running in MOCK MODE.


---
## Load the Game of Thrones Graph

**Two improvements over the naive approach:**
1. **Batch loading with `UNWIND`** — one network round-trip per entity type instead of one per record. Over Aura (cloud-hosted, cross-region latency) this is the difference between ~2s and ~30s.
2. **`MERGE` on houses inside the character load** — so a character can never end up *orphaned* just because their house wasn't pre-declared.

> Database analogy: `UNWIND $rows AS row` is a **bulk `INSERT ... VALUES`** instead of a loop of single-row inserts.

In [ ]:
banner("Loading GOT Graph Data (batched)")

# ── Data (kills corrected to match the show — your learners WILL check!) ──
HOUSES = [
    {"name": "Stark",     "region": "North",        "sigil": "Direwolf"},
    {"name": "Lannister", "region": "Westerlands",  "sigil": "Lion"},
    {"name": "Baratheon", "region": "Stormlands",   "sigil": "Stag"},
    {"name": "Targaryen", "region": "Dragonstone",  "sigil": "Dragon"},
    {"name": "Greyjoy",   "region": "Iron Islands", "sigil": "Kraken"},
    {"name": "Martell",   "region": "Dorne",        "sigil": "Sun and Spear"},
    {"name": "Tyrell",    "region": "Reach",        "sigil": "Rose"},
    {"name": "Bolton",    "region": "North",        "sigil": "Flayed Man"},
    {"name": "Frey",      "region": "Riverlands",   "sigil": "Towers"},
    {"name": "Arryn",     "region": "Vale",         "sigil": "Falcon"},
    {"name": "Clegane",   "region": "Westerlands",  "sigil": "Three Dogs"},
    {"name": "Tarly",     "region": "Reach",        "sigil": "Huntsman"},
    {"name": "Dothraki",  "region": "Essos",        "sigil": "Horse"},
    {"name": "Free Folk", "region": "Beyond the Wall", "sigil": "None"},
]

CHARACTERS = [
    {"name": "Eddard Stark",       "house": "Stark",     "alive": False, "title": "Lord of Winterfell"},
    {"name": "Catelyn Stark",      "house": "Stark",     "alive": False, "title": ""},
    {"name": "Robb Stark",         "house": "Stark",     "alive": False, "title": "King in the North"},
    {"name": "Sansa Stark",        "house": "Stark",     "alive": True,  "title": "Queen in the North"},
    {"name": "Arya Stark",         "house": "Stark",     "alive": True,  "title": ""},
    {"name": "Bran Stark",         "house": "Stark",     "alive": True,  "title": "The Three-Eyed Raven"},
    {"name": "Jon Snow",           "house": "Stark",     "alive": True,  "title": "Lord Commander"},
    {"name": "Tywin Lannister",    "house": "Lannister", "alive": False, "title": "Hand of the King"},
    {"name": "Cersei Lannister",   "house": "Lannister", "alive": False, "title": "Queen Regent"},
    {"name": "Jaime Lannister",    "house": "Lannister", "alive": False, "title": "Kingslayer"},
    {"name": "Tyrion Lannister",   "house": "Lannister", "alive": True,  "title": "Hand of the Queen"},
    {"name": "Joffrey Baratheon",  "house": "Baratheon", "alive": False, "title": "King"},
    {"name": "Robert Baratheon",   "house": "Baratheon", "alive": False, "title": "King"},
    {"name": "Renly Baratheon",    "house": "Baratheon", "alive": False, "title": ""},
    {"name": "Stannis Baratheon",  "house": "Baratheon", "alive": False, "title": ""},
    {"name": "Daenerys Targaryen", "house": "Targaryen", "alive": False, "title": "Mother of Dragons"},
    {"name": "Viserys Targaryen",  "house": "Targaryen", "alive": False, "title": ""},
    {"name": "Theon Greyjoy",      "house": "Greyjoy",   "alive": False, "title": ""},
    {"name": "Euron Greyjoy",      "house": "Greyjoy",   "alive": False, "title": ""},
    {"name": "Oberyn Martell",     "house": "Martell",   "alive": False, "title": "The Red Viper"},
    {"name": "Margaery Tyrell",    "house": "Tyrell",    "alive": False, "title": ""},
    {"name": "Olenna Tyrell",      "house": "Tyrell",    "alive": False, "title": "Queen of Thorns"},
    {"name": "Ramsay Bolton",      "house": "Bolton",    "alive": False, "title": ""},
    {"name": "Roose Bolton",       "house": "Bolton",    "alive": False, "title": ""},
    {"name": "Walder Frey",        "house": "Frey",      "alive": False, "title": ""},
    {"name": "Lysa Arryn",         "house": "Arryn",     "alive": False, "title": ""},
    {"name": "Gregor Clegane",     "house": "Clegane",   "alive": False, "title": "The Mountain"},
    {"name": "Khal Drogo",         "house": "Dothraki",  "alive": False, "title": ""},
    {"name": "Ygritte",            "house": "Free Folk", "alive": False, "title": ""},
    {"name": "Samwell Tarly",      "house": "Tarly",     "alive": True,  "title": "Grand Maester"},
]

KILLS = [  # (killer, victim) — show canon
    {"killer": "Arya Stark",      "victim": "Walder Frey"},
    {"killer": "Ramsay Bolton",   "victim": "Roose Bolton"},
    {"killer": "Sansa Stark",     "victim": "Ramsay Bolton"},
    {"killer": "Roose Bolton",    "victim": "Robb Stark"},
    {"killer": "Tyrion Lannister","victim": "Tywin Lannister"},
    {"killer": "Gregor Clegane",  "victim": "Oberyn Martell"},
    {"killer": "Olenna Tyrell",   "victim": "Joffrey Baratheon"},
    {"killer": "Jon Snow",        "victim": "Daenerys Targaryen"},
    {"killer": "Euron Greyjoy",   "victim": "Rhaegal"},  # will silently no-op: not a Character — see OBSERVE below
]

MARRIAGES = [  # stored ONCE; matched undirected with -[:MARRIED_TO]-
    {"a": "Eddard Stark",       "b": "Catelyn Stark"},
    {"a": "Robert Baratheon",   "b": "Cersei Lannister"},
    {"a": "Daenerys Targaryen", "b": "Khal Drogo"},
    {"a": "Sansa Stark",        "b": "Tyrion Lannister"},
    {"a": "Ramsay Bolton",      "b": "Sansa Stark"},
]

def load_got_data():
    if not NEO4J_AVAILABLE:
        print("[MOCK] Skipping data load (Neo4j not available).")
        return

    # Constraints — IF NOT EXISTS makes these safely re-runnable
    for c in [
        "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Character) REQUIRE c.name IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (h:House)     REQUIRE h.name IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (r:Region)    REQUIRE r.name IS UNIQUE",
    ]:
        run_cypher(c, readonly=False)

    # ONE round-trip per entity type, not per row
    run_cypher("""
        UNWIND $rows AS row
        MERGE (h:House {name: row.name})
        SET h.sigil = row.sigil
        MERGE (r:Region {name: row.region})
        MERGE (h)-[:LOCATED_IN]->(r)
    """, {"rows": HOUSES}, readonly=False)

    run_cypher("""
        UNWIND $rows AS row
        MERGE (c:Character {name: row.name})
        SET c.alive = row.alive, c.title = row.title
        MERGE (h:House {name: row.house})     // MERGE, not MATCH → no orphans
        MERGE (c)-[:BELONGS_TO]->(h)
    """, {"rows": CHARACTERS}, readonly=False)

    run_cypher("""
        UNWIND $rows AS row
        MATCH (k:Character {name: row.killer})
        MATCH (v:Character {name: row.victim})
        MERGE (k)-[:KILLED]->(v)
    """, {"rows": KILLS}, readonly=False)

    run_cypher("""
        UNWIND $rows AS row
        MATCH (a:Character {name: row.a})
        MATCH (b:Character {name: row.b})
        MERGE (a)-[:MARRIED_TO]->(b)          // one direction only
    """, {"rows": MARRIAGES}, readonly=False)

    print("✅  GOT data loaded (4 batched round-trips).")

load_got_data()

In [ ]:
# ── Verify: what did we actually load? (output-first) ─────────────
if NEO4J_AVAILABLE:
    section("Graph contents")
    for row in run_cypher("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS n ORDER BY n DESC"):
        print(f"  {row['label']:<10} {row['n']} nodes")
    print()
    for row in run_cypher("MATCH ()-[r]->() RETURN type(r) AS rel, count(r) AS n ORDER BY n DESC"):
        print(f"  {row['rel']:<12} {row['n']} relationships")

    observe("The 'Euron killed Rhaegal' row loaded ZERO relationships — Rhaegal "
            "isn't a Character node, so the MATCH found nothing and the row was "
            "silently skipped. In graph loading, MATCH-based inserts fail SILENTLY. "
            "That is why the character loader uses MERGE for houses.")
else:
    print("[MOCK] Nothing to verify.")

---
## Read-Only Guardrail *(Week 2 callback)*

We are about to let an LLM write queries that run against a **live database**. Rule zero of text-to-SQL / text-to-Cypher: **the model must be physically unable to write.**

Two layers of defence:
1. **Static check** — reject any generated Cypher containing write clauses (`CREATE`, `DELETE`, `SET`, …) before it ever reaches the DB
2. **Read transaction routing** — `execute_read` makes Neo4j itself reject writes, even if the static check misses something

> Same principle as the *gated write* in your Week 6 SLA-watchdog agent: never trust the model with a destructive capability by default.

In [ ]:
WRITE_CLAUSES = re.compile(
    r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|LOAD\s+CSV|CALL\s+\{|"
    r"CALL\s+db\.|CALL\s+dbms\.|CALL\s+apoc\.(?!text|coll|convert))\b",
    re.IGNORECASE,
)

def extract_cypher(raw: str) -> str:
    """Strip markdown fences and surrounding chatter from an LLM response."""
    fenced = re.search(r"```(?:cypher)?\s*(.*?)```", raw, re.DOTALL | re.IGNORECASE)
    text = fenced.group(1) if fenced else raw
    return text.strip().rstrip(";")

def validate_readonly(cypher: str) -> Optional[str]:
    """Return an error message if the Cypher is unsafe, else None."""
    m = WRITE_CLAUSES.search(cypher)
    if m:
        return f"Blocked: write clause '{m.group(0)}' is not allowed in read-only mode."
    return None

def with_limit(cypher: str, cap: int = 50) -> str:
    """Append a LIMIT if the query has none — protects against runaway results."""
    if re.search(r"\bLIMIT\s+\d+", cypher, re.IGNORECASE):
        return cypher
    return f"{cypher}\nLIMIT {cap}"

# ── Offline sanity checks (run before the live demo!) ─────────────
assert validate_readonly("MATCH (n) DETACH DELETE n") is not None
assert validate_readonly("MATCH (n) SET n.x = 1") is not None
assert validate_readonly("MERGE (n:Evil)") is not None
assert validate_readonly("MATCH (n:Character) RETURN n.name") is None
assert extract_cypher("```cypher\nMATCH (n) RETURN n\n```") == "MATCH (n) RETURN n"
assert "LIMIT 50" in with_limit("MATCH (n) RETURN n")
assert with_limit("MATCH (n) RETURN n LIMIT 5").count("LIMIT") == 1
print("✅  Guardrail checks pass")

---
## Text-to-Cypher with a Self-Correction Loop

The pipeline: **question → Cypher → guardrail → execute → (retry on error) → summarise**.

The retry is the interesting part: when the generated Cypher fails, we feed the **error message back to the LLM** and ask it to fix its own query. One retry recovers the large majority of failures in practice — this is the standard production pattern.

In [ ]:
GRAPH_SCHEMA = """
Graph schema:
- (Character {name: str, alive: bool, title: str})
- (House {name: str, sigil: str})
- (Region {name: str})
- (Character)-[:BELONGS_TO]->(House)
- (House)-[:LOCATED_IN]->(Region)
- (Character)-[:KILLED]->(Character)
- (Character)-[:MARRIED_TO]-(Character)   -- stored once; ALWAYS match undirected: -[:MARRIED_TO]-
"""

CYPHER_SYSTEM = f"""You translate questions into READ-ONLY Neo4j Cypher.
{GRAPH_SCHEMA}
Rules:
- Return ONLY the Cypher query. No explanations, no markdown fences.
- Read-only: never use CREATE, MERGE, DELETE, SET, REMOVE.
- Match MARRIED_TO without a direction arrow: -[:MARRIED_TO]-
- Use case-sensitive exact names as given in the question."""

def generate_cypher(question: str, previous_error: Optional[str] = None,
                    previous_cypher: Optional[str] = None) -> str:
    user = f"Question: {question}\nCypher:"
    if previous_error:
        user = (f"Question: {question}\n"
                f"Your previous query failed.\n"
                f"Previous query:\n{previous_cypher}\n"
                f"Error: {previous_error}\n"
                f"Write a corrected Cypher query:")
    return extract_cypher(client.chat(user=user, system=CYPHER_SYSTEM, temperature=0))

def ask_got(question: str, verbose: bool = True, max_retries: int = 1) -> str:
    """Natural language → Cypher → guardrail → execute → retry → summarise."""
    cypher, error, results = None, None, []

    for attempt in range(max_retries + 1):
        cypher = generate_cypher(question, previous_error=error, previous_cypher=cypher)
        if verbose:
            tag = " (retry)" if attempt else ""
            print(f"🔍 Generated Cypher{tag}:\n{cypher}\n")

        error = validate_readonly(cypher)          # layer 1: static guardrail
        if error:
            if verbose: print(f"🛡️  {error}\n")
            continue

        if not NEO4J_AVAILABLE:
            return "Neo4j unavailable — connect your Aura instance for live answers."

        try:
            results = run_cypher(with_limit(cypher), readonly=True)  # layer 2: read txn
            error = None
            break
        except Exception as e:
            error = str(e)
            if verbose: print(f"❌ Cypher error → feeding back to LLM: {error[:200]}\n")

    if error:
        return f"Could not answer after {max_retries + 1} attempts. Last error: {error[:200]}"
    if not results:
        return "The query ran successfully but returned no results."

    summary_prompt = f"""The user asked: "{question}"
The graph database returned:
{json.dumps(results, indent=2, default=str)}

Write a clear, direct answer. Present lists as lists; state counts explicitly."""
    return client.chat(user=summary_prompt, temperature=0.3)

In [ ]:
banner("Graph RAG Queries")

test_questions = [
    "Who killed Robb Stark?",
    "List all the living characters from House Stark.",
    "Who is married to Daenerys Targaryen?",
    "Which house has the most members?",
    "Who killed the person that Sansa Stark was married to?",   # 2-hop traversal!
]

for q in test_questions:
    print(f"\n📌 Q: {q}")
    print(f"💬 A: {ask_got(q, verbose=True)}")
    print("─" * 60)

discuss("Look at the last question — 'who killed the person Sansa was married to?' "
        "A vector search over text chunks would need one chunk that happens to "
        "contain the whole answer. The graph just walks two relationships: "
        "MARRIED_TO then KILLED. Multi-hop questions are where graph RAG wins.")

In [ ]:
# ── Guardrail in action: try to make the LLM destroy the database ──
section("Guardrail demo — an adversarial question")
print(ask_got("Delete all characters from the database.", verbose=True))

observe("Even if the LLM obliges and writes a DELETE, the static check blocks it "
        "— and even if the regex ever missed, execute_read means the database "
        "itself would refuse the write. Defence in depth.")

---
## Hybrid: Graph Facts + Text Context *(Week 4 callback)*

Real systems combine both retrievers: the **graph** contributes precise, structured facts; **text chunks** (from your Week 4 vector store) contribute nuance and narrative. The LLM merges them and cites which source said what.

In [ ]:
def hybrid_got_answer(question: str, text_context: Optional[List[str]] = None) -> str:
    graph_answer = ask_got(question, verbose=False)
    text_str = ""
    if text_context:
        text_str = "TEXT CONTEXT (unstructured, e.g. from a vector store):\n" + \
                   "\n\n".join(text_context[:2])

    merge_prompt = f"""Answer the question using BOTH sources. Cite which facts
came from the graph vs the text.

GRAPH ANSWER (structured relationships):
{graph_answer}

{text_str}

Question: {question}
Answer:"""
    return client.chat(user=merge_prompt, temperature=0.3)

sample_text = """Robb Stark, the King in the North, was betrayed at the Red Wedding,
an event orchestrated by Walder Frey and Tywin Lannister. Roose Bolton personally
delivered the fatal blow, whispering 'The Lannisters send their regards.'"""

banner("Hybrid Query: Who killed Robb Stark, and why?")
print(f"💬 {hybrid_got_answer('Who killed Robb Stark, and why?', text_context=[sample_text])}")

discuss("The graph knew WHO (Roose Bolton → Robb Stark). Only the text knew WHY "
        "(the Red Wedding betrayal). Neither source alone answers the full question.")

---
## Bonus: Direct Cypher (no LLM)

The graph is queryable without any LLM — useful for fixed dashboards and (Week 6 callback) as deterministic **agent tools**, like the blast-radius traversal in your change-risk assessor.

In [ ]:
banner("Direct Cypher Queries")

section("Most Deadly Characters")
for row in run_cypher("""
    MATCH (k:Character)-[:KILLED]->(v:Character)
    RETURN k.name AS killer, count(v) AS kills
    ORDER BY kills DESC LIMIT 5"""):
    print(f"  {row['killer']}: {row['kills']} kill(s)")

section("Houses by Member Count")
for row in run_cypher("""
    MATCH (c:Character)-[:BELONGS_TO]->(h:House)
    RETURN h.name AS house, count(c) AS members
    ORDER BY members DESC LIMIT 5"""):
    print(f"  {row['house']}: {row['members']} members")

section("Shortest Path: Arya Stark ↔ Tywin Lannister")
rows = run_cypher("""
    MATCH (a:Character {name: 'Arya Stark'}), (b:Character {name: 'Tywin Lannister'})
    MATCH path = shortestPath((a)-[*..6]-(b))
    RETURN [n IN nodes(path) | n.name] AS path""")
print("  " + " → ".join(rows[0]["path"]) if rows else "  No path found (or mock mode).")

observe("shortestPath is bounded to 6 hops ([*..6]) — unbounded variable-length "
        "patterns can explode combinatorially on larger graphs.")

In [ ]:
# ── Clean up: always close the driver ──────────────────────────────
if driver:
    driver.close()
    print("✅  Neo4j driver closed.")

---
## Summary & Bridge to Week 6

| Component | What it demonstrated |
| :--- | :--- |
| **Batched `UNWIND` load** | One round-trip per entity type — bulk INSERT, not a loop |
| **Read-only guardrail** | Static check + `execute_read` — defence in depth *(Week 2)* |
| **Self-correction loop** | Failed Cypher + error fed back to the LLM — production pattern |
| **Multi-hop question** | "Who killed the person Sansa married?" — where graphs beat vectors |
| **Hybrid mode** | Graph facts (WHO) + text context (WHY) *(Week 4)* |

**Bridge → Week 6:** today the LLM generated *one query per question*. Next week we hand it **tools** and let it *decide* which queries to run, in what order, across multiple turns — that's an **agent**. The graph traversal you saw today returns as the blast-radius tool in the change-request risk assessor.